# Supplementary Table 6

Shuffled-query controls: for each of 60 acquisitions and two shuffle seeds, the number of canonical peptide observations accepted at peptide q ≤ 0.01 and the number of protein entries selected, for the exact, bag, mass and combined lookup routes. This notebook rebuilds the table from `source_data/STab6/`, compares it with the table as published, repeats the producer's checks and prints the numbers quoted in the legend and the Supplementary Methods.

Run from the repository root: `python paper.py build STab6`.

In [ ]:
from pathlib import Path
import sys
UNIT = Path.cwd()                 # the notebook runs with its own directory as cwd
REPO = UNIT.parents[1]
sys.path[:0] = [str(REPO), str(UNIT)]
SD = REPO / "source_data" / UNIT.name
OUT = REPO / "build" / UNIT.name   # rendered panels; gitignored
OUT.mkdir(parents=True, exist_ok=True)
from style import fl_style as fl
import panels
fl.init()

## Table

480 acquisition-seed rows (60 acquisitions x 2 seeds x 4 routes) and a four-row summary per route. Supports the legend of Supplementary Table 6 (v28d line 485) and the sentence "Shuffled-query controls yielded no accepted peptides through the exact or bag route and 13,512 peptide–acquisition–seed observations through the mass route across 60 acquisitions and two seeds" (v28d line 561). Unit: acquisition-seed run; n = 60 acquisitions (15 CAMPI, 43 healthy stool, 1 CapScan, 1 PREDICT), 2 seeds each. No test; counts are sums over runs and are not globally distinct sequences.

In [ ]:
import pandas as pd
d = panels.load(SD)
acq = panels.build_acquisition_seed_rows(d)
summary = panels.build_summary(acq)
paths = panels.write_table(acq, summary, OUT)
print(acq.shape)
print(summary.to_string())
print(panels.cohort_mix(d).to_string())
acq.head(8)

## Producer checks

The producer's own checks, repeated on the source tables: 60 receipts of 80 rows; every receipt complete and labelled workflow FDR unverified; the accounting identities of `finish.py`; selected protein counts independent of q and level; 12 Sage result files per receipt; the `package_release.py` assertions on the mass and exact totals.

In [ ]:
chk, ok = panels.checks(d)
chk.to_csv(OUT / "CHECKS.tsv", sep="\t", index=False)
print(chk.to_string(index=False))
print("ALL_CHECKS_PASS" if ok else "CHECK_FAILURE")
assert ok

## Checks against the table as produced and recomputed numbers

The two TSVs must be byte-identical to `expected/`; the workbook is compared cell by cell (its bytes carry a timestamp). Receipts go to `checks/`. Then every number in `NUMBERS.tsv` is printed in order.

In [ ]:
import hashlib, json
def sha(p): return hashlib.sha256(Path(p).read_bytes()).hexdigest()
exp = UNIT / "expected"
receipt = {}
for k, name in (("by_acq", panels.TSV_BY_ACQ), ("summary", panels.TSV_SUMMARY)):
    receipt[name] = {"new_sha256": sha(paths[k]), "expected_sha256": sha(exp / name)}
    receipt[name]["identical"] = receipt[name]["new_sha256"] == receipt[name]["expected_sha256"]
receipt[panels.XLSX] = panels.compare_xlsx_values(paths["xlsx"], exp / panels.XLSX)
receipt[panels.XLSX]["expected_sha256"] = sha(exp / panels.XLSX)
receipt["result"] = "IDENTICAL" if all(v["identical"] for v in receipt.values()) else "DIFFERENT"
(UNIT / "checks").mkdir(exist_ok=True)
(UNIT / "checks" / "table.json").write_text(json.dumps(receipt, indent=2) + "\n")
print(json.dumps(receipt, indent=2))
assert receipt["result"] == "IDENTICAL"

print("NUMBERS.tsv order")
print("485 acquisitions:", panels.stat_n_acquisitions(d))
print("485 seeds:", panels.word(panels.stat_n_seeds(d)))
print("485 q cutoff:", panels.stat_q_cutoff(d))
print("485 exact accepted:", panels.word(panels.stat_accepted(d, "exact")))
print("485 bags accepted:", panels.word(panels.stat_accepted(d, "bags")))
print("485 mass accepted:", panels.stat_accepted(d, "mass"))
print("485 combined accepted:", panels.stat_accepted(d, "combined"))
for m in panels.METHODS:
    print(f"485 {m} selected protein entries:", panels.stat_selected_proteins(d, m))
print("561 mass accepted:", panels.stat_accepted(d, "mass"))
print("561 acquisitions:", panels.stat_n_acquisitions(d))
print("561 seeds:", panels.word(panels.stat_n_seeds(d)))

## Current Supplementary Table 5: all 726 acquisitions
The preceding cells preserve the historical 60-acquisition control. The v42 table includes all 726 acquisitions and both shuffle seeds; exact and bag controls are nonzero in one acquisition–seed.

In [ ]:
from tables.STab6 import current
current_path = current.write(SD, OUT)
acq_current, summary_current = current.load(SD)
display(summary_current)
print(current.numbers(SD))
current_check = panels.compare_xlsx_values(current_path, UNIT / "expected" / current.XLSX, float_columns={"Acquisition_seed_rows": {"ratio_to_exact"}})
assert current_check["identical"], current_check
# Verify current registered claims from the underlying acquisition/seed rows.
import csv, re
from tools.compare_values import same_numeric_value
vals = current.numbers(SD)
for row in csv.DictReader((UNIT / "NUMBERS.tsv").open(), delimiter="\t"):
    key = re.search(r"numbers\[(.+)\]", row["v42_code"]).group(1)
    assert same_numeric_value(vals[key], row["v42_recomputed_value"]), (key, vals[key])
print("Current table and all registered claims reproduced")
